# 03 — Scaling Analysis (ChestX-ray14, Cross-Polytope)


In [ ]:
import numpy as np
import pandas as pd
import pathlib
import glob
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')


In [ ]:
BENCHMARK_DIR = pathlib.Path('../results/benchmark')
FIGURES_DIR = pathlib.Path('../results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_and_merge(prefix):
    files = sorted(BENCHMARK_DIR.glob(f'{prefix}_chest_*_seed*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

df_qual = load_and_merge('benchmark_quality')
df_time = load_and_merge('benchmark_timing')
df_cand = load_and_merge('benchmark_candidates')
df_mem = load_and_merge('benchmark_memory')
print(df_qual['base_size'].unique() if not df_qual.empty else 'no data')


In [ ]:
if not df_qual.empty and not df_time.empty and not df_cand.empty:
    merge_keys = ['dataset', 'base_size', 'split_seed', 'hash_seed', 'method']
    df_merged = (df_qual.merge(df_time, on=merge_keys)
                         .merge(df_cand, on=merge_keys)
                         .merge(df_mem, on=merge_keys))
    df_agg = df_merged.groupby(['base_size', 'method'])[
        ['recall@10', 'median_latency_s', 'mean_cand_fraction', 'build_time_s']
    ].mean().reset_index()
    print(df_agg)


In [ ]:
if not df_agg.empty:
    plt.figure(figsize=(9, 6))
    order = sorted(df_agg['base_size'].astype(float).unique())
    sns.lineplot(data=df_agg, x='base_size', y='median_latency_s', hue='method', marker='o')
    plt.yscale('log')
    plt.title('Median query latency vs base size (ChestX-ray14)')
    plt.xlabel('Base size'); plt.ylabel('Median latency (s)')
    plt.tight_layout(); plt.savefig(FIGURES_DIR / 'fig7_scaling_latency.png', dpi=150)
    plt.show()


In [ ]:
if not df_agg.empty:
    speedup = df_agg[df_agg['method'] != 'linear'].merge(
        df_agg[df_agg['method'] == 'linear'][['base_size', 'median_latency_s']],
        on='base_size', suffixes=('', '_lin')
    )
    speedup['speedup'] = speedup['median_latency_s_lin'] / speedup['median_latency_s']
    print(speedup[['base_size', 'method', 'speedup', 'recall@10']])


### Scaling Summary

1. **Latency scaling**: CrossPolytope query cost grows with the candidate
   set, which is a *fraction* of the base — sublinear vs the O(n·d) linear scan.
2. **Recall**: must stay >= 0.95 (calibration constraint) across base sizes.
3. **Candidate fraction**: the key trade-off — smaller candidates ⇒ faster queries.
